# XGBoost. Очистка рядов и калибровка

Проверяем обучение на очищенной цели и шести статусах. Очистка выполняется до признаков. Сравниваем все 48 получасовых стартов и девять стартов с 08:30 до 12:30. Для каждого режима обучаются восемь прямых моделей.

Параметры `refit_params` встроены из `notebooks/xgb_tuning/best_params/best_params.json`. В Kaggle включите GPU и подключите исходные parquet.

Калибровка выбирается на val, после чего фиксируется для локального test. Метрики считаются по исходным ответам. Перед сабмитом все 16 моделей переобучаются на всей очищенной размеченной истории. Оценка test сохраняется до этого переобучения.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import math
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import xgboost as xgb
from sklearn.isotonic import IsotonicRegression
from IPython.display import display
DATA_DIR = None
CACHE_INPUT = None
OUTPUT_DIR = Path('/kaggle/working/xgb_cleaning')
DEVICE = 'cuda'
N_JOBS = 4
SEED = 42
TEST_DAYS = 7
TRAIN_FRACTION = 0.8
TARGET_SCALE = 100000.0
OBJECTIVE = 'reg:absoluteerror'
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
START_MINUTES = list(range(510, 751, 30))
VARIANTS = ['all_halfhours', 'nine_starts']
CALIBRATIONS = ['none', 'global', 'grouped', 'nonlinear']
CLEANING = {'window_days': 28, 'min_days': 14, 'z_threshold': 4.0, 'log_scale_floor': 0.25, 'mad_normalization': 1.4826}
GROUP_SHRINKAGE_ROWS = 2000.0
ISOTONIC_BINS = 128
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
PIPELINE_VERSION = 'xgb_raw_cleaning_log_slot_mad_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_DIR / 'runs' / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
BEST_PARAMS = {'1': {'n_estimators': 2530, 'max_depth': 6, 'learning_rate': 0.036760715868196514, 'min_child_weight': 1.0809355847970479, 'subsample': 0.9003563537429966, 'colsample_bytree': 0.6639976846136139, 'reg_alpha': 9.305271303855188e-05, 'reg_lambda': 7.596418498005285}, '2': {'n_estimators': 1419, 'max_depth': 4, 'learning_rate': 0.10190438495585248, 'min_child_weight': 1.8389812995413264, 'subsample': 0.6817515181500764, 'colsample_bytree': 0.6619351860505892, 'reg_alpha': 0.0022965441587741537, 'reg_lambda': 14.08596091970978}, '3': {'n_estimators': 1930, 'max_depth': 3, 'learning_rate': 0.07005359250881395, 'min_child_weight': 2.2064897603579636, 'subsample': 0.8779096005107611, 'colsample_bytree': 0.7846752865583575, 'reg_alpha': 0.018954349619526665, 'reg_lambda': 1.043977626067855}, '4': {'n_estimators': 1827, 'max_depth': 4, 'learning_rate': 0.02925325002470326, 'min_child_weight': 1.5841239345284504, 'subsample': 0.8126045473410459, 'colsample_bytree': 0.7066552862291448, 'reg_alpha': 2.3192129195348187e-05, 'reg_lambda': 0.22202924731915144}, '5': {'n_estimators': 1387, 'max_depth': 3, 'learning_rate': 0.03164978915749773, 'min_child_weight': 1.9902815543275585, 'subsample': 0.6844098448217815, 'colsample_bytree': 0.7432566543439275, 'reg_alpha': 0.010252186188681373, 'reg_lambda': 0.5427730704771981}, '6': {'n_estimators': 1566, 'max_depth': 3, 'learning_rate': 0.023401679328514332, 'min_child_weight': 3.1806793320017555, 'subsample': 0.87035767368126, 'colsample_bytree': 0.7980364720685608, 'reg_alpha': 0.02123901077609503, 'reg_lambda': 0.3328783942599951}, '7': {'n_estimators': 1136, 'max_depth': 5, 'learning_rate': 0.0330759240922086, 'min_child_weight': 3.8692684642092345, 'subsample': 0.6794525496919193, 'colsample_bytree': 0.6799389555259473, 'reg_alpha': 2.7314479492664567, 'reg_lambda': 0.3796668381307971}, '8': {'n_estimators': 534, 'max_depth': 4, 'learning_rate': 0.09417045025581995, 'min_child_weight': 13.418857993305474, 'subsample': 0.689895840619612, 'colsample_bytree': 0.6918995986906583, 'reg_alpha': 0.3746751674810787, 'reg_lambda': 1.8540896283895405}}
PARAMETER_SOURCE_SHA256 = '855a594510b647bca29fa318d610f05d91823b3946fcf04e1326ecd793a579bc'
SOURCE_NOTEBOOK_SHA256 = '13ceb3fb2fb2d62b024f76fd942d9a8bb73fff884aabd9dc0f537bc36e6d8ca4'


In [ ]:
import ctypes
import warnings
if DEVICE != 'cuda':
    raise ValueError('Этот ноутбук допускает обучение только на GPU: DEVICE должен быть cuda')
if not xgb.build_info().get('USE_CUDA', False):
    raise RuntimeError('Установленная сборка XGBoost не поддерживает CUDA. Нужна CUDA-сборка XGBoost в Kaggle')
try:
    cuda_driver = ctypes.CDLL('libcuda.so.1')
    cuda_status = cuda_driver.cuInit(0)
    gpu_count = ctypes.c_int()
    if cuda_status != 0 or cuda_driver.cuDeviceGetCount(ctypes.byref(gpu_count)) != 0 or gpu_count.value < 1:
        raise RuntimeError('CUDA не видит GPU. Включите GPU accelerator в настройках Kaggle Notebook')
except OSError as error:
    raise RuntimeError('Драйвер NVIDIA недоступен. Включите GPU accelerator в Kaggle') from error
warnings.filterwarnings('error', message='(?s).*(No visible GPU|not compiled with CUDA).*', category=UserWarning)
print(f'CUDA готова. GPU: {gpu_count.value}. XGBoost: {xgb.__version__}')


In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False))
    temporary.replace(path)
if DATA_DIR is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('train_solo_track.parquet') if (p.parent / 'test_solo_track.parquet').is_file()]
    if not candidates and Path('data/train_solo_track.parquet').is_file():
        candidates = [Path('data')]
    if len(candidates) != 1:
        raise ValueError(f'Укажите DATA_DIR: найдено {len(candidates)} папок с train/test')
    DATA_DIR = candidates[0]
DATA_DIR = Path(DATA_DIR)
TRAIN_PATH = DATA_DIR / 'train_solo_track.parquet'
TEST_PATH = DATA_DIR / 'test_solo_track.parquet'
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if history.isna().any().any() or history.duplicated(['timestamp', 'route_id']).any():
    raise ValueError('Пропуски или дубликаты в исходной истории')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)) or len(history) != len(times) * len(routes):
    raise ValueError('Нарушена полная получасовая панель')
end = times[-1]
if (end.hour, end.minute) != (10, 30):
    raise ValueError('История должна заканчиваться в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
actual = pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])
if test.isna().any().any() or not test.id.is_unique or (not actual.is_unique) or (len(actual) != len(expected)) or len(expected.difference(actual)):
    raise ValueError('Некорректные конкурсные запросы')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) < TEST_DAYS + 20:
    raise ValueError('Недостаточно дат')
before_test = dates[:-TEST_DAYS]
split_index = int(math.floor(len(before_test) * TRAIN_FRACTION))
val_start = before_test[split_index].normalize()
test_start = dates[-TEST_DAYS].normalize()
val_dates = dates[(dates >= val_start) & (dates < test_start)]
calibration_select_start = val_dates[len(val_dates) // 2].normalize()
raw_target = history[TARGET].to_numpy(dtype=np.float32).reshape(len(times), len(routes)).copy()
route_levels = np.median(raw_target[times < val_start], axis=0)
route_groups = pd.DataFrame({'route_id': routes, 'volume_group': pd.qcut(pd.Series(route_levels).rank(method='first'), 5, labels=False).to_numpy()})
route_groups.to_csv(RUN_DIR / 'route_volume_groups_train.csv', index=False)
display(pd.DataFrame([{'split': 'train', 'before': str(val_start)}, {'split': 'calibration_fit', 'from': str(val_start), 'before': str(calibration_select_start)}, {'split': 'calibration_select', 'from': str(calibration_select_start), 'before': str(test_start)}, {'split': 'test', 'from': str(test_start), 'through': str(dates[-1])}]))


## Очистка до признаков

Фиксированный робастный rolling z-score применяется в `log1p`-масштабе отдельно для маршрута, поля и получасового слота суток. Медиана и MAD берутся из предыдущих 28 дней этого же слота, минимум 14 наблюдений; текущая точка исключена. MAD - медиана абсолютных отклонений наблюдений окна от медианы этого же окна. Робастный масштаб равен `max(1.4826 × MAD, 0.25)`, порог - 4. Отмеченные верхние и нижние всплески заменяются `expm1` исторической медианы. Первые 14 дней каждого слота остаются исходными. Параметры не перебираются.

Статистики окна используют предыдущие исходные наблюдения. Нет центрированных окон, интерполяции из будущего или удаления временных строк. На val/test очистка также использует только уже наблюдённую историю; факты test для оценки сохраняются исходными. Нулевые значения тоже проверяются правилом нижнего выброса; число замен нулей выводится отдельно. Такое правило может сглаживать реальные редкие события - эффект оценивается по исходному таргету.


In [ ]:
CLEANING_IMPLEMENTATION_SHA256 = '61c115a14e0e53d0f948ff5239b7147331261315748f3f997e992b989059876d'
def clean_raw_array(array, timestamps, settings):
    array = np.asarray(array, dtype=np.float32)
    if not np.isfinite(array).all() or (array < 0).any():
        raise ValueError('Для log1p нужны конечные неотрицательные исходные значения')
    cleaned = array.copy()
    flagged = np.zeros(array.shape, dtype=bool)
    slots = timestamps.hour * 2 + timestamps.minute // 30
    for slot in range(48):
        positions = np.flatnonzero(slots == slot)
        logged = np.log1p(array[positions].astype(np.float64))
        window_days = settings['window_days']
        min_days = settings['min_days']
        if not 1 <= min_days <= window_days:
            raise ValueError('Нужно 1 <= min_days <= window_days')
        if len(positions) <= min_days:
            continue
        padded = np.pad(logged, ((window_days, 0), (0, 0)), constant_values=np.nan)
        windows = np.lib.stride_tricks.sliding_window_view(padded, window_days, axis=0)[:-1]
        eligible = np.arange(len(positions)) >= min_days
        past = windows[eligible]
        median = np.nanmedian(past, axis=-1)
        mad = np.nanmedian(np.abs(past - median[..., None]), axis=-1)
        scale = np.maximum(settings['mad_normalization'] * mad, settings['log_scale_floor'])
        mask = np.abs(logged[eligible] - median) > settings['z_threshold'] * scale
        destination = positions[eligible]
        cleaned[destination] = np.where(mask, np.expm1(median), array[destination]).astype(np.float32)
        flagged[destination] = mask
    return (cleaned, flagged)
values = {}
cleaning_rows = []
for name in [TARGET, *STATUSES]:
    raw_array = history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes))
    cleaned, flagged = clean_raw_array(raw_array, times, CLEANING)
    values[name] = cleaned
    for split, mask in [('train', times < val_start), ('val', (times >= val_start) & (times < test_start)), ('test_and_final_history', times >= test_start)]:
        marks = flagged[mask]
        cleaning_rows.append({'column': name, 'split': split, 'rows': int(marks.size), 'replaced': int(marks.sum()), 'share': float(marks.mean()), 'zeros_replaced': int((marks & (raw_array[mask] == 0)).sum())})
    positions, columns = np.nonzero(flagged)
    sample = pd.DataFrame({'timestamp': times[positions[:1000]], 'route_id': routes[columns[:1000]], 'raw': raw_array[positions[:1000], columns[:1000]], 'clean': cleaned[positions[:1000], columns[:1000]]})
    sample.to_csv(RUN_DIR / f'cleaning_sample_{name}.csv', index=False)
    del cleaned, flagged, raw_array
cleaning_report = pd.DataFrame(cleaning_rows)
cleaning_report.to_csv(RUN_DIR / 'cleaning_report.csv', index=False)
display(cleaning_report)
del history


In [ ]:
FEATURE_CODE = "\n\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\ntarget = values[TARGET]\n\ndef past(array, positions):\n    positions = np.asarray(positions, dtype=np.int64)\n    valid = (positions >= 0) & (positions < len(array))\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\n    result[~valid] = np.nan\n    return result\n\ndef average_available(matrices):\n    total = np.zeros_like(matrices[0], dtype=np.float32)\n    count = np.zeros_like(total)\n    for matrix in matrices:\n        valid = np.isfinite(matrix)\n        total += np.where(valid, matrix, 0)\n        count += valid\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\n\ndef rolling(array, steps):\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\nlevel24 = rolling(target, 48)\npositions = np.arange(len(times))\nextra_signals = {}\nfor name in STATUSES[:3]:\n    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)\nextra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))\nextra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))\nfor name in STATUSES[3:]:\n    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)\nextra_signals['hub_buffer'] = values['status_5'] - values['status_4']\nextra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']\nextra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)\nhub_signal = extra_signals['status_6_diff_hourly']\n\ndef common_features(origins):\n    common = {}\n    for k in (0, 1, 2, 4):\n        common[f'target_back_{k}'] = past(target, origins - k)\n    for name, array in values.items():\n        for steps in (4, 12):\n            common[f'{name}_mean_{steps}'] = means[name, steps][origins]\n        if name in STATUSES:\n            common[f'{name}_latest'] = array[origins]\n            if name in STATUSES[:3]:\n                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)\n    for name, array in extra_signals.items():\n        common[name] = array[origins]\n    for lag in (48, 96):\n        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)\n    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)\n    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)\n    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]\n    recent = means[TARGET, 4][origins]\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\n    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\n    return common\nCORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']\nFEATURE_GROUPS = {'calendar': ['hour', 'minute', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': ['status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'], 'hub_combinations': ['status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'], 'historical_lags': ['target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48'], 'target_change': ['target_acceleration'], 'correlation_lag': CORRELATION_COLUMNS, 'pilot_start_time': ['start_sin', 'start_cos'], 'pilot_peaks': ['peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']}\n\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\n    grid = pd.DatetimeIndex(timestamps)\n    route_ids = np.asarray(route_ids)\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\n        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\n        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')\n    if len(np.unique(route_ids)) != len(route_ids):\n        raise ValueError('Идентификаторы маршрутов повторяются')\n    steps = float(max_lookback_hours) * 2\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\n        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\n        raise ValueError('min_pairs должен быть целым числом >=3')\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\n        raise ValueError('Некорректные параметры выбора корреляции')\n    max_lag = int(steps)\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))\n    anchors = np.arange(max_lag + 2, stop)\n    best_corr = np.full(len(route_ids), np.nan)\n    best_lag = np.full(len(route_ids), np.nan)\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\n    for start in range(0, len(route_ids), route_block_size):\n        finish = min(start + route_block_size, len(route_ids))\n        x = target_values[anchors, start:finish].astype(np.float64)\n        for lag in range(2, max_lag + 1):\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\n            valid = np.isfinite(x) & np.isfinite(z)\n            count = valid.sum(axis=0)\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\n            divisor = np.maximum(count, 1)\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\n            denominator = np.sqrt(vx * vz)\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\n            corr = np.clip(corr, -1, 1)\n            previous = best_corr[start:finish]\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\n            best_corr[start:finish][better] = corr[better]\n            best_lag[start:finish][better] = lag\n            best_pairs[start:finish][better] = count[better]\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\n    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})\n\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\n    if h not in range(1, 9) or chunk_rows < 1:\n        raise ValueError('Некорректный горизонт или размер блока')\n    by_route = lag_table.set_index('route_id').reindex(routes)\n    route_index = pd.Index(routes)\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\n    for name in CORRELATION_COLUMNS:\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\n    for start in range(0, len(frame), chunk_rows):\n        finish = min(start + chunk_rows, len(frame))\n        block = frame.iloc[start:finish]\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\n        route = route_index.get_indexer(block.route_id)\n        if (origin < 0).any() or (route < 0).any():\n            raise ValueError('Строка признаков не сопоставлена с историей')\n        selected = lag[route]\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\n        available &= np.isfinite(signal)\n        for name, array in quality.items():\n            frame.iloc[start:finish, column_positions[name]] = array[route]\n        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)\n        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)\n    return frame\n\ndef horizon_dataset(h, origins):\n    slot = origins + h\n    dates = times[origins] + h * STEP\n    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})\n    for name, matrix in common_features(origins).items():\n        frame[name] = matrix.ravel()\n    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}\n    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)\n    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)\n    for name, array in calendar.items():\n        frame[name] = np.repeat(array, len(routes))\n    frame = add_correlation_features(frame, h, validation_lag_table)\n    for days in (1, 2, 7):\n        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()\n    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\n    profile = np.zeros((len(origins), len(routes)))\n    counts = np.zeros_like(profile)\n    for day in range(1, 29):\n        level = past(level24, origins - day * 48)\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\n        profile += np.nan_to_num(adjusted)\n        counts += np.isfinite(adjusted)\n    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\n    if h == 1:\n        lower = np.zeros_like(target[origins])\n        upper = past(target, origins - 2)\n        for position in (origins - 1, origins):\n            total = past(target, position)\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\n        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()\n        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\n    else:\n        frame['overlap'] = np.nan\n    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])\n    return frame.astype({name: 'float32' for name in numeric})\n\nbase_horizon_dataset = horizon_dataset\nnetwork_values = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None] for name, array in values.items() if name in [TARGET, *STATUSES[:3]]}\n\ndef ratio_features(current, usual):\n    denominator = np.maximum(usual, np.float32(1.0))\n    ratio = np.divide(current, denominator, out=np.full_like(current, np.nan), where=np.isfinite(usual))\n    return ratio, np.where(np.isfinite(usual), usual < 1.0, np.nan).astype(np.float32)\n\ndef horizon_dataset(h, origins):\n    frame = base_horizon_dataset(h, origins)\n    added = {}\n    slot = origins + h\n    profile = {}\n    for offset in (-2, -1, 0, 1, 2):\n        profile[offset] = average_available([past(target, slot + offset - day * 48) for day in range(1, 8)])\n        if offset != 0:\n            added[f'slot_daily7_neighbor_{offset:+d}'] = profile[offset].ravel()\n    added['slot_daily7_change_2h'] = (profile[2] - profile[-2]).ravel()\n    stack = np.stack([profile[offset] for offset in (-2, -1, 0, 1, 2)])\n    finite = np.isfinite(stack)\n    masked = np.where(finite, stack, -np.inf)\n    peak = masked.max(axis=0)\n    available = finite.any(axis=0)\n    added['slot_daily7_peak_value'] = np.where(available, peak, np.nan).ravel()\n    added['slot_daily7_peak_offset_minutes'] = np.where(available, (masked.argmax(axis=0) - 2) * 30, np.nan).ravel()\n    for name in STATUSES[:3]:\n        for suffix, array in [('latest', values[name]), ('mean_4', means[name, 4])]:\n            current = array[origins]\n            usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\n            ratio, small = ratio_features(current, usual)\n            added[f'{name}_{suffix}_vs_usual7'] = (current - usual).ravel()\n            added[f'{name}_{suffix}_ratio_usual7'] = ratio.ravel()\n            added[f'{name}_{suffix}_usual7_small'] = small.ravel()\n    for day in (14, 21, 28):\n        added[f'slot_days_back_{day}'] = past(target, slot - day * 48).ravel()\n    added['slot_week1_minus_week2'] = (past(target, slot - 7 * 48) - past(target, slot - 14 * 48)).ravel()\n    recent = average_available([past(target, slot - day * 48) for day in (1, 2, 3)])\n    previous = average_available([past(target, slot - day * 48) for day in (4, 5, 6, 7)])\n    added['slot_recent3_minus_previous4'] = (recent - previous).ravel()\n    for name, array in network_values.items():\n        current = array[origins]\n        usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\n        ratio, small = ratio_features(current, usual)\n        features = {'latest': current, 'change_1h': current - past(array, origins - 2),\n            'vs_usual7': current - usual, 'ratio_usual7': ratio, 'usual7_small': small}\n        for suffix, feature in features.items():\n            added[f'network_{name}_{suffix}'] = np.broadcast_to(feature, (len(origins), len(routes))).ravel()\n        local = values[name][origins]\n        share = np.divide(local, current, out=np.full_like(local, np.nan), where=current > 0)\n        past_total = past(array, origins - 2)\n        old_share = np.divide(past(values[name], origins - 2), past_total, out=np.full_like(local, np.nan), where=past_total > 0)\n        added[f'network_{name}_route_share'] = share.ravel()\n        added[f'network_{name}_route_share_change_1h'] = (share - old_share).ravel()\n    return pd.concat([frame, pd.DataFrame(added, index=frame.index, dtype=np.float32)], axis=1)\n"
exec(FEATURE_CODE, globals())


## Кэш очищенных признаков

Оба варианта используют одни и те же признаки из очищенных рядов. В кэше сохраняются все доступные получасовые старты; выбор девяти стартов выполняется маской при fit. `y` - исходный факт для метрик, `y_clean` - очищенная цель обучения. Оба поля исключены из модельных входов. Словарь корреляционных лагов для независимой оценки оценивается только до val; перед полным refit он переоценивается на всей истории. Старый неочищенный кэш несовместим. Совместимый кэш ищется в подключённых Kaggle Dataset; при необходимости задайте CACHE_INPUT. Незавершённый локальный кэш достраивается по горизонтам.


In [ ]:
validation_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=val_start, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
validation_lag_table.to_csv(RUN_DIR / 'status6_lags_train.csv', index=False)
cache_spec = {'version': PIPELINE_VERSION, 'source_sha256': file_hash(TRAIN_PATH), 'requests_sha256': file_hash(TEST_PATH), 'feature_code_sha256': hashlib.sha256(FEATURE_CODE.encode()).hexdigest(), 'cleaning': CLEANING, 'cleaning_code_sha256': CLEANING_IMPLEMENTATION_SHA256, 'val_start': str(val_start), 'test_start': str(test_start), 'lag_settings': [LAG_MAX_LOOKBACK_HOURS, LAG_MIN_PAIRS, LAG_MIN_CORRELATION]}
fingerprint = hashlib.sha256(json.dumps(cache_spec, sort_keys=True).encode()).hexdigest()
LOCAL_CACHE = OUTPUT_DIR / 'feature_cache' / fingerprint

def cache_complete(folder):
    folder = Path(folder)
    try:
        metadata = json.loads((folder / 'manifest.json').read_text())
        if metadata['fingerprint'] != fingerprint:
            return False
        for h in range(1, 9):
            record = json.loads((folder / f'h{h}.json').read_text())
            if record['fingerprint'] != fingerprint or record['history_sha256'] != file_hash(folder / f'history_h{h}.parquet') or record['future_sha256'] != file_hash(folder / f'future_h{h}.parquet'):
                return False
        return True
    except (OSError, ValueError, KeyError):
        return False
if CACHE_INPUT is not None:
    if not cache_complete(CACHE_INPUT):
        raise ValueError('CACHE_INPUT несовместим или неполон')
    CACHE_DIR = Path(CACHE_INPUT)
else:
    candidates = [LOCAL_CACHE] + [p.parent for p in Path('/kaggle/input').rglob('manifest.json')]
    CACHE_DIR = next((folder for folder in candidates if cache_complete(folder)), LOCAL_CACHE)
if CACHE_DIR == LOCAL_CACHE:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    save_json(CACHE_DIR / 'manifest.json', {'fingerprint': fingerprint, 'spec': cache_spec, 'source_end': str(end), 'routes': routes.tolist()})
print('Кэш:', CACHE_DIR)
for h in range(1, 9):
    path = CACHE_DIR / f'history_h{h}.parquet'
    future_path = CACHE_DIR / f'future_h{h}.parquet'
    record_path = CACHE_DIR / f'h{h}.json'
    try:
        record = json.loads(record_path.read_text())
        ready = record['fingerprint'] == fingerprint and record['history_sha256'] == file_hash(path) and (record['future_sha256'] == file_hash(future_path))
    except (OSError, ValueError, KeyError):
        ready = False
    if ready:
        print(f'h{h}: совместимый кэш найден')
        continue
    origins = np.arange(len(times) - h)
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, len(origins), 24):
            block = origins[start:start + 24]
            frame = horizon_dataset(h, block)
            frame['y'] = raw_target[block + h].ravel()
            frame['y_clean'] = target[block + h].ravel()
            table = pa.Table.from_pandas(frame, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    future = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp == end + h * STEP]
    future = requests.merge(future, on=['route_id', 'timestamp'], how='left', validate='one_to_one')
    if not future.cutoff.eq(end).all():
        raise ValueError('Ошибка будущих признаков')
    future.to_parquet(future_path, index=False)
    save_json(record_path, {'fingerprint': fingerprint, 'history_sha256': file_hash(path), 'future_sha256': file_hash(future_path), 'rows': len(origins) * len(routes)})
    print(f'h{h}: {len(origins) * len(routes):,} строк')
for name in ['values', 'means', 'level24', 'extra_signals', 'network_values', 'frame', 'table', 'future', 'raw_target']:
    globals().pop(name, None)
gc.collect()
save_json(RUN_DIR / 'fixed_settings.json', {'source_notebook_sha256': SOURCE_NOTEBOOK_SHA256, 'parameter_source_sha256': PARAMETER_SOURCE_SHA256, 'params': BEST_PARAMS, 'cleaning': CLEANING, 'feature_fingerprint': fingerprint, 'val_start': str(val_start), 'calibration_select_start': str(calibration_select_start), 'test_start': str(test_start), 'test_used_for_selection': False})


In [ ]:
def metric_row(frame, prediction, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    error = np.asarray(prediction, dtype=np.float64) - actual
    total = actual.sum()
    if total <= 0 or not np.isfinite(error).all():
        raise ValueError('Метрика неопределена: нулевая сумма цели или неконечный прогноз')
    daily = pd.DataFrame({'day': frame.cutoff.dt.normalize(), 'actual': actual, 'error': error, 'absolute': np.abs(error)}).groupby('day').sum()
    daily = daily.loc[daily.actual > 0]
    wape = np.abs(error).sum() / total
    bias = error.sum() / total
    return {'split': split, 'rows': len(frame), 'MAE': float(np.abs(error).mean()), 'WAPE': float(wape), 'Relative Bias': float(bias), '|Relative Bias|': float(abs(bias)), 'score': float(wape + abs(bias)), 'daily_score': float(((daily.absolute + daily.error.abs()) / daily.actual).mean())}

def model_columns(frame):
    return frame.columns.drop(['cutoff', 'timestamp', 'y', 'y_clean', 'overlap']).tolist()

def start_mask(frame, variant):
    if variant == 'all_halfhours':
        return np.ones(len(frame), dtype=bool)
    minutes = frame.cutoff.dt.hour * 60 + frame.cutoff.dt.minute
    return minutes.isin(START_MINUTES).to_numpy()

def make_model(h):
    return xgb.XGBRegressor(**BEST_PARAMS[str(h)], objective=OBJECTIVE, eval_metric='mae', tree_method='hist', device=DEVICE, enable_categorical=True, max_cat_to_onehot=1, random_state=SEED, n_jobs=N_JOBS)

def require_cuda(model):
    actual = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']
    if not actual.startswith('cuda'):
        raise RuntimeError(f'Обучение перешло на {actual}')
route_dtype = pd.CategoricalDtype(categories=routes)
evaluation_parts = {variant: [] for variant in VARIANTS}
model_metric_parts = []
feature_columns = {}
for h in range(1, 9):
    data = pd.read_parquet(CACHE_DIR / f'history_h{h}.parquet')
    data['route_id'] = data.route_id.astype(route_dtype)
    columns = model_columns(data)
    feature_columns[h] = columns
    train_available = data.timestamp.lt(val_start).to_numpy()
    eval_mask = data.cutoff.isin(dates[dates >= val_start]).to_numpy()
    train1030 = train_available & data.cutoff.isin(dates[dates < val_start]).to_numpy()
    for variant in VARIANTS:
        mask = train_available & start_mask(data, variant)
        model = make_model(h)
        model.fit(data.loc[mask, columns], data.loc[mask, 'y_clean'] / TARGET_SCALE, verbose=False)
        require_cuda(model)
        model.save_model(RUN_DIR / f'{variant}_h{h}_before_refit.ubj')
        train_pred = np.maximum(model.predict(data.loc[mask, columns]).astype(np.float64) * TARGET_SCALE, 0)
        train_table = metric_row(data.loc[mask], train_pred, 'train_fit_starts_raw_target')
        part = data.loc[train1030 | eval_mask, ['cutoff', 'timestamp', 'route_id', 'y']].copy()
        part['prediction'] = np.maximum(model.predict(data.loc[train1030 | eval_mask, columns]).astype(np.float64) * TARGET_SCALE, 0)
        part['horizon'] = h
        part['split'] = np.where(part.cutoff < val_start, 'train_1030', np.where(part.cutoff < test_start, 'val', 'test'))
        rows = [train_table]
        for split in ['train_1030', 'val', 'test']:
            block = part.loc[part.split.eq(split)]
            rows.append(metric_row(block, block.prediction, split))
        table = pd.DataFrame(rows)
        table.insert(0, 'horizon', h)
        table.insert(0, 'variant', variant)
        model_metric_parts.append(table)
        display(table)
        evaluation_parts[variant].append(part)
        part.to_parquet(RUN_DIR / f'{variant}_h{h}_evaluation.parquet', index=False)
        del model, train_pred
        gc.collect()
    del data
    gc.collect()
model_metrics = pd.concat(model_metric_parts, ignore_index=True)
model_metrics.to_csv(RUN_DIR / 'model_metrics_before_refit.csv', index=False)
save_json(RUN_DIR / 'feature_columns.json', feature_columns)
evaluation = {variant: pd.concat(parts, ignore_index=True) for variant, parts in evaluation_parts.items()}
del evaluation_parts


## Четыре калибровки

1. `none`: исходный прогноз.
2. `global`: сумма исходных целей / сумма прогнозов.
3. `grouped`: отдельный множитель для горизонта × квинтиля объёма маршрута. Квинтили определены по исходной train-истории. Групповой множитель стягивается к общему с фиксированными 2 000 псевдостроками.
4. `nonlinear`: общая монотонная кусочно-линейная функция прогноза. Изотоническая регрессия обучается на не более чем 128 квантильных корзинах; веса - число строк. За границами обученного диапазона сохраняется отношение крайнего откалиброванного значения к исходному, чтобы не обрезать большие прогнозы до константы.

Сравнение выполняется по общей WAPE + |Relative Bias| на поздней половине val, отдельно для двух вариантов. Калибраторы на этом шаге видят только раннюю половину val. Затем все четыре переоцениваются на полном val, а тип победителя остаётся зафиксированным. Ни XGBoost, ни параметры очистки/калибровки не перебираются. Таблицы test для всех калибровок диагностические и не меняют выбор.


In [ ]:
def calibration_keys(frame):
    mapping = route_groups.set_index('route_id').volume_group
    group = frame.route_id.astype('int64').map(mapping)
    if group.isna().any():
        raise ValueError('Маршрут без группы объёма')
    return frame.horizon.astype(str) + ':' + group.astype(int).astype(str)

def fit_calibrator(frame, method):
    actual = frame.y.to_numpy(dtype=np.float64)
    pred = frame.prediction.to_numpy(dtype=np.float64)
    if len(frame) == 0 or pred.sum() <= 0:
        raise ValueError('Недостаточно прогнозов для калибровки')
    global_factor = float(actual.sum() / pred.sum())
    result = {'method': method, 'global_factor': global_factor}
    if method == 'grouped':
        groups = pd.DataFrame({'key': calibration_keys(frame).to_numpy(), 'y': actual, 'pred': pred}).groupby('key').agg(y=('y', 'sum'), pred=('pred', 'sum'), n=('y', 'size'))
        local = groups.y / groups.pred.replace(0, np.nan)
        weight = groups.n / (groups.n + GROUP_SHRINKAGE_ROWS)
        factors = (weight * local + (1 - weight) * global_factor).fillna(global_factor)
        result['factors'] = {str(k): float(v) for k, v in factors.items()}
    elif method == 'nonlinear':
        edges = np.unique(np.quantile(pred, np.linspace(0, 1, ISOTONIC_BINS + 1)))
        bins = np.searchsorted(edges[1:-1], pred, side='right')
        binned = pd.DataFrame({'bin': bins, 'x': pred / TARGET_SCALE, 'y': actual / TARGET_SCALE}).groupby('bin').agg(x=('x', 'mean'), y=('y', 'mean'), weight=('y', 'size')).sort_values('x')
        if len(binned) < 2:
            result['fallback_global'] = True
        else:
            iso = IsotonicRegression(y_min=0, out_of_bounds='clip')
            iso.fit(binned.x, binned.y, sample_weight=binned.weight)
            result['x'] = (iso.X_thresholds_ * TARGET_SCALE).tolist()
            result['y'] = (iso.y_thresholds_ * TARGET_SCALE).tolist()
    return result

def apply_calibrator(frame, calibrator):
    pred = frame.prediction.to_numpy(dtype=np.float64)
    method = calibrator['method']
    if method == 'none':
        result = pred.copy()
    elif method == 'global' or calibrator.get('fallback_global', False):
        result = pred * calibrator['global_factor']
    elif method == 'grouped':
        factor = calibration_keys(frame).map(calibrator['factors']).fillna(calibrator['global_factor']).to_numpy(dtype=np.float64)
        result = pred * factor
    elif method == 'nonlinear':
        x = np.asarray(calibrator['x'])
        y = np.asarray(calibrator['y'])
        result = np.interp(pred, x, y)
        lower = pred < x[0]
        upper = pred > x[-1]
        result[lower] = pred[lower] * (y[0] / x[0] if x[0] > 0 else calibrator['global_factor'])
        result[upper] = pred[upper] * (y[-1] / x[-1] if x[-1] > 0 else calibrator['global_factor'])
    else:
        raise ValueError(method)
    if not np.isfinite(result).all():
        raise ValueError('Неконечный результат калибровки')
    return np.maximum(result, 0)
calibrators = {}
selected_calibrations = {}
selection_rows = []
for variant, part in evaluation.items():
    fit_part = part.loc[part.split.eq('val') & part.cutoff.lt(calibration_select_start)]
    select_part = part.loc[part.split.eq('val') & part.cutoff.ge(calibration_select_start)]
    variant_rows = []
    for method in CALIBRATIONS:
        fitted = fit_calibrator(fit_part, method)
        pred = apply_calibrator(select_part, fitted)
        row = {'variant': variant, 'calibration': method, **metric_row(select_part, pred, 'calibration_select')}
        variant_rows.append(row)
    selected_calibrations[variant] = min(variant_rows, key=lambda row: row['score'])['calibration']
    selection_rows.extend(variant_rows)
    val_part = part.loc[part.split.eq('val')]
    calibrators[variant] = {method: fit_calibrator(val_part, method) for method in CALIBRATIONS}
selection = pd.DataFrame(selection_rows)
display(selection)
selection.to_csv(RUN_DIR / 'calibration_selection.csv', index=False)
save_json(RUN_DIR / 'calibrators.json', {'fit_period': [str(val_start), str(test_start)], 'selection_split': str(calibration_select_start), 'selected': selected_calibrations, 'calibrators': calibrators, 'group_shrinkage_rows': GROUP_SHRINKAGE_ROWS, 'isotonic_bins': ISOTONIC_BINS})
calibration_metric_rows = []
for variant, part in evaluation.items():
    for method, fitted in calibrators[variant].items():
        for split in ['train_1030', 'val', 'test']:
            block = part.loc[part.split.eq(split)]
            pred = apply_calibrator(block, fitted)
            calibration_metric_rows.append({'variant': variant, 'calibration': method, 'selected_on_val': method == selected_calibrations[variant], **metric_row(block, pred, split)})
calibration_metrics = pd.DataFrame(calibration_metric_rows)
calibration_metrics.to_csv(RUN_DIR / 'independent_calibrated_metrics_before_refit.csv', index=False)
display(calibration_metrics)
print('Выбор только на val:', selected_calibrations)
print('Независимая от выбора по val оценка test получена до переобучения. Даты уже изучались в проекте.')


## Полное переобучение

Все 16 моделей заново обучаются на всей доступной очищенной размеченной истории: восемь для всех получасов и восемь для девяти стартов. Число деревьев фиксировано из `refit_params`, early stopping отсутствует. Статистический лаг переоценивается на полной истории. Причинный rolling-фильтр не имеет отдельного fit на будущем и остаётся тем же. Калибраторы, обученные на исходных фактах val и прогнозах моделей до refit, фиксируются; по in-sample прогнозам полной модели они не переобучаются. Test после refit не оценивается.


In [ ]:
full_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=end + STEP, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
full_lag_table.to_csv(RUN_DIR / 'status6_lags_full.csv', index=False)
future_parts = {variant: [] for variant in VARIANTS}
full_rows = []
for h in range(1, 9):
    data = add_correlation_features(pd.read_parquet(CACHE_DIR / f'history_h{h}.parquet'), h, full_lag_table)
    future = add_correlation_features(pd.read_parquet(CACHE_DIR / f'future_h{h}.parquet'), h, full_lag_table)
    data['route_id'] = data.route_id.astype(route_dtype)
    future['route_id'] = future.route_id.astype(route_dtype)
    columns = feature_columns[h]
    for variant in VARIANTS:
        mask = start_mask(data, variant)
        model = make_model(h)
        model.fit(data.loc[mask, columns], data.loc[mask, 'y_clean'] / TARGET_SCALE, verbose=False)
        require_cuda(model)
        model.save_model(RUN_DIR / f'{variant}_h{h}_full.ubj')
        result = future[['id', 'route_id', 'timestamp', 'cutoff']].copy()
        result['horizon'] = h
        result['prediction'] = np.maximum(model.predict(future[columns]).astype(np.float64) * TARGET_SCALE, 0)
        future_parts[variant].append(result)
        pred = np.maximum(model.predict(data.loc[mask, columns]).astype(np.float64) * TARGET_SCALE, 0)
        row = {'variant': variant, 'horizon': h, **metric_row(data.loc[mask], pred, 'full_history_after_refit_raw_target')}
        full_rows.append(row)
        old = model_metrics.loc[model_metrics.variant.eq(variant) & model_metrics.horizon.eq(h)]
        display(pd.concat([old, pd.DataFrame([row])], ignore_index=True))
        del model, pred
        gc.collect()
    del data, future
    gc.collect()
pd.DataFrame(full_rows).to_csv(RUN_DIR / 'full_fit_metrics.csv', index=False)


In [ ]:
submission_rows = []
for variant in VARIANTS:
    frame = pd.concat(future_parts[variant], ignore_index=True)
    if len(frame) != len(test) or not frame.id.is_unique or set(frame.horizon.unique()) != set(range(1, 9)):
        raise ValueError('Неполный набор прогнозов')
    frame.to_parquet(RUN_DIR / f'{variant}_future_raw.parquet', index=False)
    for method in CALIBRATIONS:
        frame['y_pred'] = apply_calibrator(frame, calibrators[variant][method])
        submission = test[['id']].merge(frame[['id', 'y_pred']], on='id', how='left', validate='one_to_one')
        if len(submission) != len(test) or not submission.id.equals(test.id.reset_index(drop=True)) or (not np.isfinite(submission.y_pred).all()) or (submission.y_pred < 0).any():
            raise ValueError('Некорректный сабмит')
        path = RUN_DIR / f'submission_{variant}_{method}.csv'
        submission.to_csv(path, index=False)
        if method == selected_calibrations[variant]:
            submission.to_csv(OUTPUT_DIR / f'submission_{variant}_selected.csv', index=False)
        submission_rows.append({'variant': variant, 'calibration': method, 'selected_on_val': method == selected_calibrations[variant], 'file': str(path), 'rows': len(submission)})
submissions = pd.DataFrame(submission_rows)
submissions.to_csv(RUN_DIR / 'submission_files.csv', index=False)
display(submissions)
print('Выбранные сабмиты:', OUTPUT_DIR / 'submission_all_halfhours_selected.csv', OUTPUT_DIR / 'submission_nine_starts_selected.csv')
print('Независимая оценка test получена до полного переобучения; таблица independent_calibrated_metrics_before_refit.csv.')
